In [1]:
# =========================================================
# 0. Imports
# =========================================================
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# =========================================================
# 1. Config
# =========================================================
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B"
FINETUNED_MODEL_NAME = "businessrules/Qwen-base-br-qlora_exp17"

TEST_DATASET_NAME = "businessrules/dataset_stratified_test"
OUTPUT_DATASET_NAME = "businessrules/Qwen_base_tuned_exp17_results"
   

MAX_NEW_TOKENS = 768
MODEL_CONTEXT = 8192
MAX_PROMPT_TOKENS = MODEL_CONTEXT - MAX_NEW_TOKENS

2026-08-11 14:12:19.895120: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786457540.138980      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786457540.209389      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786457540.790323      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786457540.790371      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786457540.790374      24 computation_placer.cc:177] computation placer alr

In [2]:
import os
os.environ["HF_TOKEN"] = "hf_token" # removed for safety reasosns

from huggingface_hub import login
login(os.environ["HF_TOKEN"])

test_dataset = load_dataset(TEST_DATASET_NAME, split="test")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


README.md:   0%|          | 0.00/510 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.85M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/486k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
# =========================================================
# 2. Prompt (MUST MATCH TRAINING)
# =========================================================
def build_prompt(code):
    return f"""### Instruction:
Extract the business rules implemented by the following code.

Constraints:
- Return ONLY the business rules
- Use Markdown bullet points
- DO NOT repeat the instruction
- DO NOT repeat the code
- DO NOT add explanations
- DO NOT wrap output in code blocks
- End when the rules end

### Code:
{code}

### Business Rules:
"""


def truncate_code_to_fit(code):
    prompt_without_code = build_prompt("")
    prompt_tokens = tokenizer(prompt_without_code, return_tensors="pt")["input_ids"].shape[1]

    max_code_tokens = MAX_PROMPT_TOKENS - prompt_tokens - 10  # safety buffer

    code_tokens = tokenizer(
        code,
        return_tensors="pt",
        truncation=True,
        max_length=max_code_tokens,
    )

    return tokenizer.decode(code_tokens["input_ids"][0], skip_special_tokens=True)


# =========================================================
# 3. Tokenizer
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right" 


# =========================================================
# 4. Load FINE-TUNED model (LoRA / QLoRA compatible)
# =========================================================
ft_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

ft_model = PeftModel.from_pretrained(
    ft_model,
    FINETUNED_MODEL_NAME
)
ft_model.eval()


# =========================================================
# 5. Generation function (deterministic)
# =========================================================
@torch.no_grad()
def generate_business_rules(model, code):
    code = truncate_code_to_fit(code)
    prompt = build_prompt(code)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    )

    # DEBUG: token length check
    input_len = inputs["input_ids"].shape[1]
    print(f"Input tokens: {input_len}")

    inputs = inputs.to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,            
        temperature=1.0,
        top_p=1.0,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Robust output extraction
    if "### Business Rules:" in decoded:
        decoded = decoded.split("### Business Rules:", 1)[-1]

    return decoded.strip()


# =========================================================
# 6. Run inference (FINE-TUNED ONLY)
# =========================================================
results = []

for idx, example in enumerate(test_dataset, start=1):
    print(f"\n Processing example {idx}/{len(test_dataset)} (id={example['id']})")

    code = example["cd"]

    ft_pred = generate_business_rules(ft_model, code)

    results.append({
        "id": example["id"],
        "golden_business_rule": example["br"],
        "finetuned_model_prediction": ft_pred,
        "finetuned_model": FINETUNED_MODEL_NAME,
    })


In [ ]:
import json
import pandas as pd

# 1. Save raw JSON backup
with open("results_backup.json", "w") as f:
    json.dump(results, f)

# 2. Convert to Pandas DataFrame (safer for handling Parquet conversion)
df = pd.DataFrame(results)

# 3. Save as Parquet locally
df.to_parquet("results.parquet")

print("Safe! Data saved locally as 'results_backup.json' and 'results.parquet'")

In [ ]:
from huggingface_hub import HfApi

# Initialize API
api = HfApi()

# Define where you want to upload
repo_id = OUTPUT_DATASET_NAME 

# Create the repo if it doesn't exist yet
api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

# Upload the parquet file we saved in Step 1
api.upload_file(
    path_or_fileobj="results.parquet",
    path_in_repo="data.parquet",  
    repo_id=repo_id,
    repo_type="dataset"
)

print(f"Dataset uploaded successfully to https://huggingface.co/datasets/{repo_id}")